In [1]:
import pandas as pd

# Chargement des données INSEE du Rhône
df_insee_69 = pd.read_csv("../data/raw/insee_comparateur_territoires_69.csv")

# Variables INSEE retenues pour l'enrichissement
variables_2023 = [
    "MED_SL",
    "POP",
    "SUP",
    "DWELLINGS",
    "DWELLINGS_OCS_DW_MAIN",
    "DWELLINGS_OCS_DW_MAIN_TSH_100",
    "DWELLINGS_OCS_DW_SEC_DW_OCC",
    "DWELLINGS_OCS_DW_VAC"
]

# Sélection des indicateurs 2023
df_2023 = df_insee_69[(df_insee_69["TAB_MEASURE"].isin(variables_2023)) & (df_insee_69["TIME_PERIOD"] == 2023)].copy()

# UNIT_LOC est disponible en 2024
df_2024 = df_insee_69[(df_insee_69["TAB_MEASURE"] == "UNIT_LOC") & (df_insee_69["TIME_PERIOD"] == 2024)].copy()

# Regroupement
df_insee_selection = pd.concat([df_2023, df_2024], ignore_index=True)

print("Dimensions :", df_insee_selection.shape)
print("Nombre de communes :", df_insee_selection["GEO"].nunique())

display(df_insee_selection[["GEO", "TIME_PERIOD", "TAB_MEASURE", "OBS_VALUE"]].head(20))

Dimensions : (2475, 11)
Nombre de communes : 275


,GEO,TIME_PERIOD,TAB_MEASURE,OBS_VALUE
0,69381,2023,POP,28717.00000
1,69381,2023,SUP,1.51000
2,69381,2023,DWELLINGS_OCS_DW_MAIN_TSH_100,4918.81463
3,69381,2023,DWELLINGS,19187.70160
4,69381,2023,DWELLINGS_OCS_DW_MAIN,16414.21303
5,69381,2023,DWELLINGS_OCS_DW_SEC_DW_OCC,1008.95533
6,69381,2023,DWELLINGS_OCS_DW_VAC,1764.53324
7,69381,2023,MED_SL,28810.00000
8,69382,2023,POP,29981.00000
9,69382,2023,SUP,3.41000


In [2]:
# Passage du format long au format large
df_insee_pivot = df_insee_selection.pivot(index="GEO", columns="TAB_MEASURE", values="OBS_VALUE").reset_index()

# Supprimer le nom de l'axe des colonnes
df_insee_pivot.columns.name = None

print("Dimensions :", df_insee_pivot.shape)
display(df_insee_pivot.head())

Dimensions : (275, 10)


,GEO,DWELLINGS,DWELLINGS_OCS_DW_MAIN,DWELLINGS_OCS_DW_MAIN_TSH_100,DWELLINGS_OCS_DW_SEC_DW_OCC,DWELLINGS_OCS_DW_VAC,MED_SL,POP,SUP,UNIT_LOC
0,69001,182.00361,154.45646,138.62355,15.30397,12.24318,28280.0,398.0,10.65,9.0
1,69002,199.03115,118.69782,99.16117,43.55422,36.77912,25270.0,235.0,12.95,8.0
2,69003,1274.16244,1148.93288,648.00658,24.00233,101.22723,31060.0,3057.0,2.57,60.0
3,69004,309.11431,277.71003,228.88364,5.23405,26.17023,31600.0,777.0,3.61,13.0
4,69005,264.19653,239.68728,197.17000,1.58124,22.92801,29560.0,651.0,4.55,55.0


In [3]:
# Renommage des variables
noms_variables = {
    "GEO": "code_commune",
    "MED_SL": "niveau_vie_median",
    "POP": "population",
    "SUP": "superficie_km2",
    "DWELLINGS": "nb_logements",
    "DWELLINGS_OCS_DW_MAIN": "nb_residences_principales",
    "DWELLINGS_OCS_DW_MAIN_TSH_100": "nb_proprietaires",
    "DWELLINGS_OCS_DW_SEC_DW_OCC": "nb_residences_secondaires",
    "DWELLINGS_OCS_DW_VAC": "nb_logements_vacants",
    "UNIT_LOC": "nb_etablissements"
}

df_insee_final = df_insee_pivot.rename(columns=noms_variables)

display(df_insee_final.head())


,code_commune,nb_logements,nb_residences_principales,nb_proprietaires,nb_residences_secondaires,nb_logements_vacants,niveau_vie_median,population,superficie_km2,nb_etablissements
0,69001,182.00361,154.45646,138.62355,15.30397,12.24318,28280.0,398.0,10.65,9.0
1,69002,199.03115,118.69782,99.16117,43.55422,36.77912,25270.0,235.0,12.95,8.0
2,69003,1274.16244,1148.93288,648.00658,24.00233,101.22723,31060.0,3057.0,2.57,60.0
3,69004,309.11431,277.71003,228.88364,5.23405,26.17023,31600.0,777.0,3.61,13.0
4,69005,264.19653,239.68728,197.17000,1.58124,22.92801,29560.0,651.0,4.55,55.0


In [4]:
# Vérification des données après pivot

print("Dimensions :", df_insee_final.shape)
print("Nombre de communes :", df_insee_final["code_commune"].nunique())
print("Nombre de doublons :", df_insee_final["code_commune"].duplicated().sum())

print("\nValeurs manquantes :")
print(df_insee_final.isna().sum())

display(df_insee_final.describe())

Dimensions : (275, 10)
Nombre de communes : 275
Nombre de doublons : 0

Valeurs manquantes :
code_commune                 0
nb_logements                 0
nb_residences_principales    0
nb_proprietaires             0
nb_residences_secondaires    0
nb_logements_vacants         0
niveau_vie_median            1
population                   0
superficie_km2               0
nb_etablissements            0
dtype: int64


,code_commune,nb_logements,nb_residences_principales,nb_proprietaires,nb_residences_secondaires,nb_logements_vacants,niveau_vie_median,population,superficie_km2,nb_etablissements
count,275.000000,275.000000,275.000000,275.000000,275.000000,275.000000,274.000000,275.000000,275.000000,275.000000
mean,69156.276364,4762.050355,4171.937502,1872.001183,199.175279,390.937574,29586.277372,8850.160000,11.989055,378.381818
std,95.914176,21354.951263,18183.306628,6136.193378,1249.189211,1964.749836,4239.917324,35041.206009,8.955937,1775.191373
min,69001.000000,33.000000,22.000000,18.000000,1.581240,1.000000,18640.000000,120.000000,0.420000,3.000000
25%,69075.500000,408.019165,348.691440,274.079195,16.821480,27.827580,26855.000000,845.000000,5.980000,23.000000
50%,69153.000000,906.214710,808.890070,594.000000,28.639120,52.000000,28915.000000,1994.000000,10.110000,63.000000
75%,69234.500000,2214.183015,2015.434945,1397.995185,56.839575,139.133625,31920.000000,4780.500000,15.460000,189.000000
max,69389.000000,321273.502360,271968.490040,90817.783440,19298.208460,30006.803850,47810.000000,519127.000000,83.600000,27316.000000


In [5]:
#Création features

# Densité de population
df_insee_final["densite_population"] = (df_insee_final["population"] / df_insee_final["superficie_km2"])

# Part des propriétaires
df_insee_final["part_proprietaires"] = (df_insee_final["nb_proprietaires"] / df_insee_final["nb_residences_principales"] * 100)

# Part des logements vacants
df_insee_final["part_logements_vacants"] = (df_insee_final["nb_logements_vacants"] / df_insee_final["nb_logements"] * 100)

# Vérification
print("Dimensions :", df_insee_final.shape)
display(df_insee_final.head())

# Sauvegarde données INSEE préparées
df_insee_final.to_csv("../data/processed/insee_comparateur_territoires_69.csv", index=False)

print("Fichier enregistré : insee_comparateur_territoires_69_processed.csv")
print("Dimensions :", df_insee_final.shape)

Dimensions : (275, 13)


,code_commune,nb_logements,nb_residences_principales,nb_proprietaires,nb_residences_secondaires,nb_logements_vacants,niveau_vie_median,population,superficie_km2,nb_etablissements,densite_population,part_proprietaires,part_logements_vacants
0,69001,182.00361,154.45646,138.62355,15.30397,12.24318,28280.0,398.0,10.65,9.0,37.370892,89.749273,6.726889
1,69002,199.03115,118.69782,99.16117,43.55422,36.77912,25270.0,235.0,12.95,8.0,18.146718,83.540852,18.479077
2,69003,1274.16244,1148.93288,648.00658,24.00233,101.22723,31060.0,3057.0,2.57,60.0,1189.494163,56.400734,7.944609
3,69004,309.11431,277.71003,228.88364,5.23405,26.17023,31600.0,777.0,3.61,13.0,215.235457,82.418212,8.466198
4,69005,264.19653,239.68728,197.17000,1.58124,22.92801,29560.0,651.0,4.55,55.0,143.076923,82.261353,8.678392


Fichier enregistré : insee_comparateur_territoires_69_processed.csv
Dimensions : (275, 13)


In [6]:
print("Dimensions :", df_insee_final.shape)
print("Nombre de communes :", df_insee_final["code_commune"].nunique())
print("Nombre de doublons :", df_insee_final["code_commune"].duplicated().sum())

print("\nValeurs manquantes :")
display(df_insee_final.isna().sum())

print("\nStatistiques des nouvelles features :")
display(
    df_insee_final[
        [
            "densite_population",
            "part_proprietaires",
            "part_logements_vacants"
        ]
    ].describe().T
)

Dimensions : (275, 13)
Nombre de communes : 275
Nombre de doublons : 0

Valeurs manquantes :


code_commune                 0
nb_logements                 0
nb_residences_principales    0
nb_proprietaires             0
nb_residences_secondaires    0
nb_logements_vacants         0
niveau_vie_median            1
population                   0
superficie_km2               0
nb_etablissements            0
densite_population           0
part_proprietaires           0
part_logements_vacants       0
dtype: int64


Statistiques des nouvelles features :


,count,mean,std,min,25%,50%,75%,max
densite_population,275.0,956.620296,2419.187179,15.445619,79.803798,205.044510,622.995943,19017.880795
part_proprietaires,275.0,71.231709,13.670430,26.325168,65.417709,75.105091,80.981676,96.470591
part_logements_vacants,275.0,7.117540,3.029028,0.324675,4.922658,6.696429,8.868451,22.660148
